<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Model Inference (Segmentation)</h1>
<h4 align="left">Written by the KSO Team</h4>

This notebook runs a trained YOLO **segmentation** model on a folder of videos and produces:

- `all_detections.csv` - one row per detected instance: frame, timestamp, class, confidence, mask area fraction, bounding box (and the polygon, optionally).
- `coverage_by_frame.csv` - one row per (frame, class): the fraction of the frame covered by that class, with overlapping instances of the same class merged so coverage never exceeds 100%.
- An annotated overlay video with masks drawn on the footage (on by default).

It is the segmentation sibling of the bounding-box `Model_Inference.ipynb`. Same stream-mode, constant-memory loop. The only differences: it reads masks instead of only boxes, computes coverage, and the overlay shows masks.

### Coverage in one line
For each frame and class, coverage = (pixels covered by that class) / (total pixels in the frame). Averaging that over all frames gives "on average, X% of the view is this class". A mask is used for the area, not the bounding box, so branchy algae are not over-counted.

---

### Key parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `conf_thres` | 0.5 | Minimum detection confidence |
| `vid_stride` | 5 | Process every Nth frame (higher = faster, fewer frames) |
| `img_size` | 640 | YOLO input resolution (match training if custom) |
| `WRITE_OVERLAY` | True | Write an annotated mask-overlay video of the footage |
| `SAVE_POLYGONS` | False | Add a normalized polygon-coordinate column to `all_detections.csv` (larger file) |
| `CLEAN_GOPRO` | False | Strip GoPro metadata streams that can cause OpenCV errors |

Timestamps in the CSVs always reflect real video time regardless of stride.

---
## Phase 1: Configuration

Edit the fields below to match your setup, then run the cell. Use a `-seg` weight (for example `yolo11n-seg.pt` -> `best.pt` from a segmentation run). A detection model will produce no masks and empty coverage.

In [1]:
from pathlib import Path

# ---- Paths ----
model_path = Path("/scratch/project_465002425/ferfersv/models/LST_skane_100E_11s_1.5_conf0.001/weights/best.pt").expanduser().resolve()   # -seg weights (best.pt)
videos_dir = Path("/scratch/project_465002425/ferfersv/videos/LST_skane_video").expanduser().resolve()                            # Your video folder
output_dir = Path("/scratch/project_465002425/ferfersv/inference/skane").expanduser().resolve()                         # csv/overlay outputs

# ---- Inference ----
conf_thres = 0.3         # Minimum confidence threshold (best practice-> conf_thres=F1 score of model)
vid_stride = 1           # Process every Nth frame (1 = all)
img_size   = 640         # YOLO input size (match training if custom)

# ---- Outputs ----
WRITE_OVERLAY = True     # Annotated mask-overlay video of the full footage
SAVE_POLYGONS = True    # Store normalized polygon coords per instance (where/what); larger CSV

# ---- Options ----
n_videos = "all"         # "all" (with quotes) or an integer like 5 (without quotes)
CLEAN_GOPRO = False      # Strip metadata streams that cause OpenCV errors

# ---- Time range ----
# Trim seconds from start/end of each video before inference (requires ffmpeg).
# Keyframe-aligned copy, no re-encode, so the cut may land within ~1-2s of the request.
# CSV timestamps are rewritten to original video time.
skip_start_s = 0
skip_end_s   = 0

# ---- Validate ----
assert model_path.is_file() and model_path.suffix == ".pt", f"Model weights (.pt) not found: {model_path}"
assert n_videos == "all" or isinstance(n_videos, int), f'n_videos must be "all" or an int, got {n_videos!r}'
assert skip_start_s >= 0 and skip_end_s >= 0, "skip_start_s and skip_end_s must be >= 0"

# ---- Run identity (own folder per model/stride/conf) ----
run_id = f"{model_path.parent.parent.name}_stride{vid_stride}_conf{int(conf_thres*100)}"
run_output_dir = output_dir / run_id

print(f"Model:    {model_path.parent.parent.name}")
print(f"Videos:   {videos_dir}")
print(f"Output:   {run_output_dir}")
print(f"Overlay:  {WRITE_OVERLAY}")
print(f"Polygons: {SAVE_POLYGONS}")
print(f"Process:  {n_videos} video(s)")
print(f"Trim:     -{skip_start_s}s start, -{skip_end_s}s end")

Model:    LST_skane_100E_11s_1.5_conf0.001
Videos:   /pfs/lustrep2/scratch/project_465002425/ferfersv/videos/LST_skane_video
Output:   /pfs/lustrep2/scratch/project_465002425/ferfersv/inference/skane/LST_skane_100E_11s_1.5_conf0.001_stride1_conf30
Overlay:  True
Polygons: True
Process:  all video(s)
Trim:     -0s start, -0s end


---
## Phase 2: Run Inference

Cleans/trims (if enabled), runs segmentation, writes both CSVs incrementally (per video), and writes the overlay video.

In [2]:
import gc, time, subprocess, shutil, sys, json
import numpy as np
import torch
import cv2
import pandas as pd
from ultralytics import YOLO

# ---- Validate paths ----
assert model_path.is_file() and model_path.suffix == ".pt", f"Model weights (.pt) not found: {model_path}"
assert videos_dir.is_dir(), f"Video directory not found: {videos_dir}"

# ---- Output setup ----
run_output_dir.mkdir(parents=True, exist_ok=True)
det_csv = run_output_dir / "all_detections.csv"
cov_csv = run_output_dir / "coverage_by_frame.csv"
for f in (det_csv, cov_csv):
    if f.exists():
        f.unlink()

# ---- Discover videos ----
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
all_videos = sorted(
    p for p in videos_dir.iterdir()
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS and "_clean" not in p.stem and "_trim" not in p.stem
)
selected = all_videos[:n_videos] if isinstance(n_videos, int) else all_videos
assert len(selected) > 0, f"No video files found in {videos_dir}"
print(f"{len(selected)} video(s) in {videos_dir}")

# ---- Find ffmpeg (only if cleaning or trimming) ----
need_ffmpeg = CLEAN_GOPRO or skip_start_s > 0 or skip_end_s > 0
ffmpeg_exe = None
if need_ffmpeg:
    ffmpeg_exe = shutil.which("ffmpeg")
    if ffmpeg_exe is None:
        try:
            import imageio_ffmpeg
            ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
        except ImportError:
            pass
    if ffmpeg_exe is None:
        print("Warning: ffmpeg not found; skipping cleaning and trimming")

# ---- Load model ----
model = YOLO(str(model_path))
names = model.names
half = torch.cuda.is_available()
if model.task != "segment":
    print(f"Warning: model task is '{model.task}', not 'segment'. "
          f"Coverage needs a -seg model; mask outputs will be empty.")
print(f"Model: {model_path.name} | Task: {model.task} | Classes: {names}")
print("=" * 60)

total_dets = 0
total_frames_all = 0
det_header_written = False
cov_header_written = False

for idx, video_path in enumerate(selected, 1):

    # ---- Pre-process (clean and/or trim) ----
    # Working copies live at output_dir/working_videos/ (shared across runs).
    inference_path = video_path
    trim_offset = 0.0

    if ffmpeg_exe and (CLEAN_GOPRO or skip_start_s > 0 or skip_end_s > 0):
        work_dir = output_dir / "working_videos"
        work_dir.mkdir(exist_ok=True)

        parts = []
        if CLEAN_GOPRO:
            parts.append("clean")
        if skip_start_s > 0 or skip_end_s > 0:
            parts.append(f"trim{int(skip_start_s)}_{int(skip_end_s)}")
        worked = work_dir / f"{video_path.stem}_{'_'.join(parts)}.mp4"

        if not worked.exists():
            cmd = [ffmpeg_exe]
            if skip_start_s > 0:
                cmd += ["-ss", str(skip_start_s)]
            cmd += ["-i", str(video_path)]
            if skip_end_s > 0:
                cap_t = cv2.VideoCapture(str(video_path))
                fps_t = cap_t.get(cv2.CAP_PROP_FPS) or 30.0
                n_t = int(cap_t.get(cv2.CAP_PROP_FRAME_COUNT))
                cap_t.release()
                duration_s = n_t / fps_t if fps_t > 0 else 0
                keep = duration_s - skip_start_s - skip_end_s
                if keep > 0:
                    cmd += ["-t", str(keep)]
            cmd += ["-map", "0:v:0", "-c", "copy", "-an", "-sn", "-map_metadata", "-1", "-y", str(worked)]
            r = subprocess.run(cmd, capture_output=True)
            if r.returncode == 0:
                inference_path = worked
                trim_offset = float(skip_start_s)
            else:
                print(f"  ffmpeg failed, using original: {r.stderr.decode()[:200]}")
        else:
            inference_path = worked
            trim_offset = float(skip_start_s)

    # ---- Video metadata ----
    cap = cv2.VideoCapture(str(inference_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    expected = max(1, total_frames // max(vid_stride, 1))
    frame_px = float(w * h)
    print(f"\n[{idx}/{len(selected)}] {video_path.name}  ({w}x{h}, {fps:.1f}fps, {total_frames/fps:.1f}s)")

    # ---- Overlay writer ----
    overlay_writer = None
    overlay_path = None
    if WRITE_OVERLAY:
        overlay_dir = run_output_dir / "overlay"
        overlay_dir.mkdir(exist_ok=True)
        overlay_path = overlay_dir / f"{video_path.stem}_seg.mp4"
        overlay_fps = max(1.0, fps / max(vid_stride, 1))   # keeps real-time duration

    # ---- Inference loop ----
    det_rows = []
    cov_rows = []
    frame_count = 0
    start = time.time()

    for result in model.predict(
        source=str(inference_path), conf=conf_thres, stream=True,
        vid_stride=vid_stride, imgsz=img_size, half=half, verbose=False,
    ):
        src_frame = frame_count * vid_stride
        ts = src_frame / fps + trim_offset   # Original video time

        masks = result.masks
        boxes = result.boxes
        if masks is not None and boxes is not None and len(boxes):
            clss = boxes.cls.int().cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            xyxy = boxes.xyxy.cpu().numpy()
            polys = masks.xy   # list of (N,2) arrays in original-frame pixels

            # One row per instance (where it is, what it is, how big)
            for poly, cf, c, b in zip(polys, confs, clss, xyxy):
                area = cv2.contourArea(poly.astype(np.float32)) if len(poly) >= 3 else 0.0
                row = {
                    "video": video_path.stem,
                    "frame": src_frame,
                    "timestamp_s": round(ts, 3),
                    "class_name": names[int(c)],
                    "confidence": round(float(cf), 4),
                    "area_frac": round(area / frame_px, 6),
                    "x1": round(float(b[0]), 1), "y1": round(float(b[1]), 1),
                    "x2": round(float(b[2]), 1), "y2": round(float(b[3]), 1),
                }
                if SAVE_POLYGONS:
                    pn = (poly / np.array([w, h], dtype=np.float32)).round(5)
                    row["polygon_norm"] = json.dumps(pn.flatten().tolist())
                det_rows.append(row)

            # Per-frame, per-class coverage (instances merged, no double-count)
            for c in np.unique(clss):
                canvas = np.zeros((h, w), dtype=np.uint8)
                n_inst = 0
                for poly, cl in zip(polys, clss):
                    if cl == c and len(poly) >= 3:
                        cv2.fillPoly(canvas, [poly.astype(np.int32)], 1)
                        n_inst += 1
                cov_rows.append({
                    "video": video_path.stem,
                    "frame": src_frame,
                    "timestamp_s": round(ts, 3),
                    "class_name": names[int(c)],
                    "coverage_frac": round(float(canvas.sum()) / frame_px, 6),
                    "n_instances": int(n_inst),
                })

        # Overlay: result.plot() draws the masks automatically (no manual drawing)
        if WRITE_OVERLAY:
            ann = result.plot(line_width=2, labels=True, conf=True)
            if overlay_writer is None:
                fh, fw = ann.shape[:2]
                overlay_writer = cv2.VideoWriter(
                    str(overlay_path), cv2.VideoWriter_fourcc(*"mp4v"), overlay_fps, (fw, fh)
                )
            overlay_writer.write(ann)

        frame_count += 1

        if frame_count % 200 == 0:
            elapsed = time.time() - start
            pct = min(frame_count / expected * 100, 100)
            rate = frame_count / elapsed
            eta = (expected - frame_count) / rate if rate > 0 else 0
            sys.stdout.write(f"\r  {pct:5.1f}% | {frame_count}/{expected} frames | {rate:.0f} fps | ETA {eta:.0f}s  ")
            sys.stdout.flush()

    if overlay_writer:
        overlay_writer.release()

    # ---- Append this video's rows to CSVs (incremental) ----
    if det_rows:
        pd.DataFrame(det_rows).to_csv(det_csv, mode="a", header=not det_header_written, index=False)
        det_header_written = True
    if cov_rows:
        pd.DataFrame(cov_rows).to_csv(cov_csv, mode="a", header=not cov_header_written, index=False)
        cov_header_written = True

    total_dets += len(det_rows)
    total_frames_all += frame_count
    elapsed = time.time() - start
    rate = frame_count / max(elapsed, 0.1)
    sys.stdout.write(f"\r  {frame_count} frames | {elapsed:.1f}s | {rate:.0f} fps | {len(det_rows)} detections")
    if WRITE_OVERLAY and overlay_path is not None:
        sys.stdout.write(f" | overlay: {overlay_path.name}")
    print()

    del det_rows, cov_rows
    gc.collect()

# ---- Summary ----
print("\n" + "=" * 60)
print(f"{len(selected)} video(s) | {total_frames_all} frames inferred | {total_dets} detections")
if det_csv.exists() and total_dets > 0:
    print("\nDetections by class:")
    print(pd.read_csv(det_csv, usecols=["class_name"])["class_name"].value_counts().to_string())
if cov_csv.exists() and total_frames_all > 0:
    cov = pd.read_csv(cov_csv, usecols=["class_name", "coverage_frac"])
    mean_cov = cov.groupby("class_name")["coverage_frac"].sum() / total_frames_all
    print("\nMean coverage across all frames (frames where a class is absent count as 0):")
    print((mean_cov.sort_values(ascending=False) * 100).round(2).astype(str).add(" %").to_string())
print(f"\n-> {det_csv}")
print(f"-> {cov_csv}")

1 video(s) in /pfs/lustrep2/scratch/project_465002425/ferfersv/videos/LST_skane_video
Model: best.pt | Task: segment | Classes: {0: 'Algal turf', 1: 'Filamentous algae', 2: 'Fucus sp.'}

[1/1] GX010009_fixed.mp4  (3840x2160, 30.0fps, 543.8s)
  16296 frames | 2073.9s | 8 fps | 77250 detections | overlay: GX010009_fixed_seg.mp4

1 video(s) | 16296 frames inferred | 77250 detections

Detections by class:
class_name
Algal turf           33484
Fucus sp.            33352
Filamentous algae    10414

Mean coverage across all frames (frames where a class is absent count as 0):
class_name
Algal turf           40.99 %
Fucus sp.            38.06 %
Filamentous algae     4.65 %

-> /pfs/lustrep2/scratch/project_465002425/ferfersv/inference/skane/LST_skane_100E_11s_1.5_conf0.001_stride1_conf30/all_detections.csv
-> /pfs/lustrep2/scratch/project_465002425/ferfersv/inference/skane/LST_skane_100E_11s_1.5_conf0.001_stride1_conf30/coverage_by_frame.csv


---
## Done

```
<output_dir>/<model>_stride<N>_conf<C>/
  all_detections.csv         one row per instance: frame, timestamp_s, class_name,
                             confidence, area_frac, x1, y1, x2, y2 [, polygon_norm]
  coverage_by_frame.csv      one row per (frame, class): coverage_frac, n_instances
  overlay/
    <video>_seg.mp4          masks drawn on the footage
working_videos/              cleaned/trimmed copies (if CLEAN_GOPRO or trim used)
```

### Columns
- `area_frac` - that single instance mask as a share of the frame (0 to 1).
- `coverage_frac` - share of the frame covered by the class in that frame, overlapping instances merged (0 to 1, never above 1).
- `polygon_norm` - present only if `SAVE_POLYGONS = True`. Flat list of normalized x,y vertices.

### Next steps
1. Feed `coverage_by_frame.csv` to the post-processing notebook for the figures (mean coverage per class, coverage over time, share by class).
2. Compare models: change `model_path`, keep the same `output_dir`, re-run. Each model/stride/conf gets its own folder.